## 1) Imports and Setup
In this section, we import the required libraries and define helper functions for evaluation.


In [22]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


## 2) Load Dataset
We load `data.csv` and verify the dataset structure.


In [23]:
BASE_DIR = Path(".")
data_path = BASE_DIR / "data.csv"

df = pd.read_csv(data_path)
df.head()


,amount,category,day,items
0,12000,Food,1,3
1,8000,Transport,2,1
2,15000,Food,3,4
3,5000,Bills,4,1
4,20000,Shopping,5,2


## 3) Define Features and Target
We separate the target column from the feature columns and then split the dataset into train and test sets.


In [24]:
target_col = "amount"  # Change this if your target column has a different name

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)


X_train shape: (5, 3)
X_test shape : (2, 3)


## 4) Detect Columns and Prepare Baseline Data
We detect categorical (text) and numeric columns.  
The baseline model cannot train on text columns, so we will use numeric features only for the baseline.


In [25]:
# Detect categorical and numeric columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns     :", num_cols)

# Baseline uses numeric features only
X_train_num = X_train[num_cols].copy()
X_test_num  = X_test[num_cols].copy()

print("Baseline X_train_num shape:", X_train_num.shape)
print("Baseline X_test_num shape :", X_test_num.shape)


Categorical columns: ['category']
Numeric columns     : ['day', 'items']
Baseline X_train_num shape: (5, 2)
Baseline X_test_num shape : (2, 2)


## 5) Baseline Model (Numeric Features Only)
We train a basic Linear Regression model using numeric features only, then evaluate MAE, RMSE, and R2.


In [26]:
baseline_model = LinearRegression()
baseline_model.fit(X_train_num, y_train)

y_pred_base = baseline_model.predict(X_test_num)

base_mae = mean_absolute_error(y_test, y_pred_base)
base_rmse = np.sqrt(mean_squared_error(y_test, y_pred_base))
base_r2 = r2_score(y_test, y_pred_base)

base_mae, base_rmse, base_r2


(1421.875000000001, np.float64(1534.9127112803521), 0.41101074218749967)

## 6) Model with One-Hot Encoding (Categorical Features)
We apply One-Hot Encoding to categorical columns and train Linear Regression again.
This allows the model to learn from text features like `category`.


In [27]:
# Build transformers safely (works even if there are no categorical columns)
transformers = []

if len(cat_cols) > 0:
    transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))

transformers.append(("num", "passthrough", num_cols))

preprocessor_onehot = ColumnTransformer(transformers=transformers)

model_encoded = Pipeline(steps=[
    ("preprocess", preprocessor_onehot),
    ("regressor", LinearRegression())
])

model_encoded.fit(X_train, y_train)
y_pred_enc = model_encoded.predict(X_test)

enc_mae = mean_absolute_error(y_test, y_pred_enc)
enc_rmse = np.sqrt(mean_squared_error(y_test, y_pred_enc))
enc_r2 = r2_score(y_test, y_pred_enc)

enc_mae, enc_rmse, enc_r2


(656.9767441860631, np.float64(735.8269631049528), 0.8646396700919355)

## 7) Optional Model: One-Hot Encoding + Scaling
We optionally scale numeric columns using StandardScaler, then train the model again.
Scaling can help when numeric features have very different ranges.


In [28]:
transformers_scaled = []

if len(cat_cols) > 0:
    transformers_scaled.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))

# Scale numeric columns
transformers_scaled.append(("num", StandardScaler(), num_cols))

preprocessor_scaled = ColumnTransformer(transformers=transformers_scaled)

model_scaled = Pipeline(steps=[
    ("preprocess", preprocessor_scaled),
    ("regressor", LinearRegression())
])

model_scaled.fit(X_train, y_train)
y_pred_scaled = model_scaled.predict(X_test)

sc_mae = mean_absolute_error(y_test, y_pred_scaled)
sc_rmse = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
sc_r2 = r2_score(y_test, y_pred_scaled)

sc_mae, sc_rmse, sc_r2


(846.7011642949483, np.float64(923.7505449128312), 0.7866712326933119)

## 8) Metrics Comparison Table (Before vs After)
We compare the performance of:
- Baseline model (numeric only)
- One-Hot encoded model
- One-Hot + Scaling model


In [29]:
results = pd.DataFrame([
    {"Model": "Baseline (Numeric Only)", "MAE": base_mae, "RMSE": base_rmse, "R2": base_r2},
    {"Model": "OneHot Encoding",         "MAE": enc_mae,  "RMSE": enc_rmse,  "R2": enc_r2},
    {"Model": "OneHot + Scaling",        "MAE": sc_mae,   "RMSE": sc_rmse,   "R2": sc_r2},
])

results_sorted = results.sort_values(by="RMSE", ascending=True)
results_sorted


,Model,MAE,RMSE,R2
1,OneHot Encoding,656.976744,735.826963,0.864640
2,OneHot + Scaling,846.701164,923.750545,0.786671
0,Baseline (Numeric Only),1421.875000,1534.912711,0.411011


## 9) Save Metrics Table
We export the comparison results to a CSV file for documentation.


In [30]:
metrics_path = Path("metrics_comparison.csv")
results_sorted.to_csv(metrics_path, index=False)

print("Saved metrics table to:", metrics_path.resolve())


Saved metrics table to: D:\courses\En + Ai + Github\Chat GPT course\Smart_Expenses_Project\labs\11_regression\metrics_comparison.csv


## 10) Save the Best Model
We select the model with the lowest RMSE and save it as a joblib file.


In [31]:
best_row = results_sorted.iloc[0]
best_model_name = best_row["Model"]

if best_model_name == "Baseline (Numeric Only)":
    best_model = baseline_model
elif best_model_name == "OneHot Encoding":
    best_model = model_encoded
else:
    best_model = model_scaled

model_path = Path("best_linear_regression_model.joblib")
joblib.dump(best_model, model_path)

print("Best model:", best_model_name)
print("Saved best model to:", model_path.resolve())


Best model: OneHot Encoding
Saved best model to: D:\courses\En + Ai + Github\Chat GPT course\Smart_Expenses_Project\labs\11_regression\best_linear_regression_model.joblib


## 11) Short Explanation (Why Metrics Changed)

- If encoding improved results:
  One-hot encoding helped the model capture category-level differences, which improved prediction accuracy.

- If scaling improved results:
  Scaling balanced numeric feature ranges, making training more stable.

- If performance decreased:
  The dataset may be small or noisy, and the new transformations might not add useful signal.
